In [1]:
# ===========================================
# Cell 1: Google Drive Mount
# ===========================================
from google.colab import drive

drive.mount('/content/drive')
print("Drive mount done!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mount done!


In [3]:
# ===========================================
# Cell 2: Config + Path Setup
# ===========================================
import os
import yaml

CONFIG_PATH = "/content/config.yaml"

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT  = f"/content/drive/MyDrive/{config['project_name']}"
DATA_PATH   = os.path.join(DRIVE_ROOT, config['paths']['processed_data'])
MODEL_PATH  = os.path.join(DRIVE_ROOT, config['paths']['models'])
REPORT_PATH = os.path.join(DRIVE_ROOT, config['paths']['outputs'], "reports")

os.makedirs(MODEL_PATH,  exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)

RANDOM_STATE = config['model']['random_state']   # 42
TEST_SIZE    = config['model']['test_size']       # 0.2

print(f"Project : {config['project_name']}")
print(f"Data    : {DATA_PATH}")

Project : EcoTracing
Data    : /content/drive/MyDrive/EcoTracing/data/processed


In [4]:
# ===========================================
# Cell 3: Library Import + EnergyMLP Class
# ===========================================
import numpy as np
import pandas as pd
import pickle
import json
import time
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing  import StandardScaler


class EnergyMLP(nn.Module):
    """
    Residual 학습용 MLP
    - 저장된 기존 MLP와 동일 구조 (network + BatchNorm1d)
    - 이번엔 y_residual을 타겟으로 학습

    Input : cpu, memory, duration (3개)
    Output : residual (energy_kwh - formula_kwh)
    """
    def __init__(self, input_size=3, hidden_sizes=[512, 256, 128], dropout=0.0):
        super().__init__()
        layers = []
        prev = input_size
        for h in hidden_sizes:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU()
            ]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device : {DEVICE}")
print("Libraries loaded!")

Device : cuda
Libraries loaded!


In [5]:
# ===========================================
# Cell 4: Load Data + Residual 계산
# ===========================================
parquet_path = os.path.join(DATA_PATH, "instance_usage_full_processed.parquet")
df = pd.read_parquet(parquet_path)
print(f"Total rows: {len(df):,}")

# 공식으로 y_formula 재계산
# Power(W) = 200 + cpu*300 + memory*50
# Energy(kWh) = Power * (duration_sec / 3600) / 1000
df['y_formula'] = (200 + df['cpu'] * 300 + df['memory'] * 50) * (df['duration'] / 3600) / 1000

# Residual = 실제값 - 공식값
# 데이터가 공식으로 만들어졌으므로 residual ≈ 0
df['residual'] = df['energy_kwh'] - df['y_formula']

print(f"\n[Residual 통계]")
print(f" mean : {df['residual'].mean():.8f}")
print(f" std : {df['residual'].std():.8f}")
print(f" min : {df['residual'].min():.8f}")
print(f" max  {df['residual'].max():.8f}")
# residual이 거의 0에 수렴하는지 확인


Total rows: 19,523,808

[Residual 통계]
 mean : 0.00000000
 std : 0.00000000
 min : 0.00000000
 max  0.00000000


In [6]:
FEATURES = ['cpu', 'memory', 'duration']
TARGET   = 'residual'

X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# 공식 기반 예측값도 test용으로 저장 (최종 y_final 계산용)
_, df_test = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE)
y_formula_test = df_test['y_formula'].values
y_actual_test  = df_test['energy_kwh'].values

# X만 정규화 (y는 residual이 ≈0이라 정규화 불필요)
scaler_X = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)

print(f"Train : {X_train_sc.shape}")
print(f"Test  : {X_test_sc.shape}")

Train : (15619046, 3)
Test  : (3904762, 3)


In [7]:
EPOCHS     = 30
BATCH_SIZE = 1024
LR         = 1e-4

train_ds = TensorDataset(
    torch.tensor(X_train_sc, dtype=torch.float32),
    torch.tensor(y_train,    dtype=torch.float32)
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

model = EnergyMLP(input_size=3, hidden_sizes=[512, 256, 128]).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

print(f"Training Residual MLP ({EPOCHS} epochs)...")
start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X_b)
        loss = criterion(pred, y_b)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(X_b)
    epoch_loss /= len(train_ds)
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:3d} | Loss: {epoch_loss:.8f}")

elapsed = time.time() - start
print(f"\nTraining done! ({elapsed:.1f}s)")

Training Residual MLP (30 epochs)...
  Epoch   5 | Loss: 0.00000023
  Epoch  10 | Loss: 0.00000008
  Epoch  15 | Loss: 0.00000006
  Epoch  20 | Loss: 0.00000005
  Epoch  25 | Loss: 0.00000004
  Epoch  30 | Loss: 0.00000003

Training done! (5250.3s)


In [8]:
model.eval()
X_test_t = torch.tensor(X_test_sc, dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    residual_pred = model(X_test_t).cpu().numpy()

# y_final = y_formula + residual_pred
y_final = y_formula_test + residual_pred

rmse = np.sqrt(mean_squared_error(y_actual_test, y_final))
mae = mean_absolute_error(y_actual_test, y_final)
r2  = r2_score(y_actual_test, y_final)
mape = np.mean(np.abs((y_actual_test - y_final) / (y_actual_test + 1e-10))) * 100

print(f"\n[Residual MLP 성능] (y_final = y_formula + MLP_residual)")
print(f" RMSE : {rmse:.8f} kWh")
print(f" MAE : {mae:.8f} kWh")
print(f" R2  : {r2:.8f}")
print(f" MAPE : {mape:.4f}%")

# 잔차 예측값 분포 확인
print(f"\n[MLP가 예측한 residual 통계]")
print(f" mean : {residual_pred.mean():.8f}")
print(f" std : {residual_pred.std():.8f}")


[Residual MLP 성능] (y_final = y_formula + MLP_residual)
 RMSE : 0.00038112 kWh
 MAE : 0.00036158 kWh
 R2  : 0.99630718
 MAPE : 69.6742%

[MLP가 예측한 residual 통계]
 mean : -0.00035333
 std : 0.00014285


In [9]:
# CPU=0.5, Memory=0.3, 1시간 / 공식 기준 0.365 kWh
FORMULA_KWH = 0.365
TEST_CPU = 0.5
TEST_MEM = 0.3
TEST_DUR_SEC = 3600.0

# 공식 계산
power_w  =200 + TEST_CPU * 300 + TEST_MEM * 50
y_formula = power_w * (TEST_DUR_SEC / 3600) / 1000 # = 0.365 kWh

# MLP residual 예측
X_infer = np.array([[TEST_CPU, TEST_MEM, TEST_DUR_SEC]], dtype=np.float32)
X_infer_sc = scaler_X.transform(X_infer)
X_infer_t = torch.tensor(X_infer_sc, dtype=torch.float32).to(DEVICE)

model.eval()
with torch.no_grad():
    residual_out = model(X_infer_t).cpu().numpy()[0]

y_final_out = y_formula + residual_out
error_pct  = abs(y_final_out - FORMULA_KWH) / FORMULA_KWH * 100

print(f"\n[Streamlit 조건 테스트] CPU=50%, Mem=30%, 1시간")
print(f" 공식 기준값 : {FORMULA_KWH:.4f} kWh")
print(f" y_formula  : {y_formula:.6f} kWh")
print(f" MLP residual 예측 : {residual_out:.8f} kWh")
print(f" y_final : {y_final_out:.6f} kWh")
print(f" 공식 대비 오차 : {error_pct:.4f}%")
print(f"\n [이전 MLP 단독 오차: 13.2%]")


[Streamlit 조건 테스트] CPU=50%, Mem=30%, 1시간
 공식 기준값 : 0.3650 kWh
 y_formula  : 0.365000 kWh
 MLP residual 예측 : 0.00119208 kWh
 y_final : 0.366192 kWh
 공식 대비 오차 : 0.3266%

 [이전 MLP 단독 오차: 13.2%]


In [10]:
# MLP state_dict 저장
mlp_file = os.path.join(MODEL_PATH, config['model']['model_names']['mlp_residual'])
torch.save(model.state_dict(), mlp_file)

# scaler_X 저장
scaler_file = os.path.join(MODEL_PATH, config['model']['model_names']['scaler_x_residual'])
with open(scaler_file, 'wb') as f:
    pickle.dump(scaler_X, f)

print(f"Saved MLP     : {mlp_file}")
print(f"Saved ScalerX : {scaler_file}")

# 결과 저장
results = {
    'method'     : 'Residual MLP (y_final = y_formula + MLP_pred)',
    'concept'    : 'MLP learns residual (y_actual - y_formula). Since data is formula-derived, residual≈0. At inference, y_formula dominates.',
    'architecture': f'3 -> 512 -> 256 -> 128 -> 1 (BatchNorm1d)',
    'epochs'     : EPOCHS,
    'batch_size' : BATCH_SIZE,
    'lr'         : LR,
    'features'   : FEATURES,
    'target'     : 'residual (energy_kwh - y_formula)',
    'metrics'    : {
        'rmse': float(rmse),
        'mae' : float(mae),
        'r2'  : float(r2),
        'mape': float(mape),
    },
    'streamlit_test': {
        'cpu'          : TEST_CPU,
        'memory'       : TEST_MEM,
        'duration_h'   : 1.0,
        'formula_kwh'  : FORMULA_KWH,
        'y_formula'    : float(y_formula),
        'residual_pred': float(residual_out),
        'y_final'      : float(y_final_out),
        'error_pct'    : float(error_pct),
    }
}

result_file = os.path.join(REPORT_PATH, config['model']['results']['results_json_residual'])
with open(result_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Results saved : {result_file}")
print("Done!")

Saved MLP     : /content/drive/MyDrive/EcoTracing/models/energy_model_mlp_residual.pkl
Saved ScalerX : /content/drive/MyDrive/EcoTracing/models/scaler_x_residual.pkl
Results saved : /content/drive/MyDrive/EcoTracing/outputs/reports/phase1_full_results_residual.json
Done!
